In [157]:
import os
import time
from pathlib import Path
import pandas as pd
import numpy as np

load the data and lets take a look

In [158]:
raw = pd.read_csv('club_signups.csv')
raw.head()

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes


Target =>
| COLUMN     | ALLOWED VALUES                                     |
| ---------- | -------------------------------------------------- |
| `faculty`  | `MET`, `IET`, `EMS`, `Pharmacy`, `Management`      |
| `club`     | `Robotics`, `Debate`, `Music`, `Football`, `Chess` |
| `city`     | `Cairo`, `Giza`, `Alexandria`                      |
| `fee_paid` | `True`, `False` (boolean, not text)                |

# Part A — Inconsistencies

1. **Initial inspection:** Display the shape, data types (`dtypes`), and `value_counts()` of every categorical column so that inconsistencies are visible before making any changes.

2. **Faculty, club, and city:** Map every variant to its canonical value. First, normalise mechanically by stripping whitespace and converting values to lowercase. Then, use an explicit dictionary to handle the remaining variants. For example:
   - `Media Engineering and Technology` → `MET`
   - `Soccer` → `Football`

   Assert that only canonical values remain in these columns.

3. **Names and emails:** Standardise names by removing extra spaces and applying consistent capitalisation. Trim whitespace from email addresses and convert them to lowercase.

4. **`fee_paid`:** Convert inconsistent representations of boolean values into a proper boolean column:
   - `yes`, `Y`, `TRUE`, `1` → `True`
   - `no`, `N`, `false`, `0` → `False`

5. **`signed_up_at`:** Identify the two date formats mixed in this column. Determine which component represents the day and which represents the month, then parse all values into a single datetime column.

# Part B — Duplicates

1. **Exact duplicates:** Count rows representing identical form submissions, then remove the duplicate copies.

2. **Repeated sign-ups:** Some students submitted sign-ups for the same club more than once, using different `signup_id` values. These resubmissions typically indicate that a student returned to update their fee payment status. Keep only the latest submission for each combination of `student_id` and `club`. Count the number of rows removed.

3. **Validation:** Assert that the combination of `student_id` and `club` is unique after completing Part B.

=============================================================================================================================

1. **Initial inspection:** Display the shape, data types (`dtypes`), and `value_counts()` of every categorical column so that inconsistencies are visible before making any changes.


In [159]:
raw.describe()

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
count,39,39,39,39,39,39,39,39,39
unique,36,15,24,18,13,9,13,36,9
top,S015,61-7344,Laila Mostafa,ali.ibrahim@student.guc.edu.eg,Pharmacy,Cairo,Robotics,2026-09-15 13:39,1
freq,2,6,4,5,7,13,8,2,7


In [160]:
from IPython.display import display, Markdown

cat_cols = ['faculty', 'city', 'club']
for col in cat_cols:
    c = raw[col]

    display(Markdown(f"## {col}"))
    display(Markdown(
        f"**Type:** {type(c).__name__} "
        f"**Shape:** {c.shape} "
        f"**Dtype:** {c.dtype}"
    ))

    display(Markdown("**Original value counts**"))
    display(
        c.value_counts()
         .rename_axis(col)
         .reset_index(name="count")
    )

    display(Markdown("**Lowercase value counts**"))
    display(
        c.str.lower()
         .value_counts(dropna=False)
         .rename_axis(col)
         .reset_index(name="count")
    )



## faculty

**Type:** Series **Shape:** (39,) **Dtype:** object

**Original value counts**

,faculty,count
0,Pharmacy,7
1,EMS,7
2,IET,5
3,Management,4
4,Information Engineering and Technology,4
5,iet,3
6,MET,3
7,Engineering and Materials Science,1
8,Pharma,1
9,Media Engineering and Technology,1


**Lowercase value counts**

...

**Type:** Series **Shape:** (39,) **Dtype:** object

**Original value counts**

,club,count
0,Robotics,8
1,Music,5
2,Chess,4
3,football,4
4,Debating,3
5,Football,3
6,Chess Club,3
7,Soccer,2
8,music,2
9,chess,2


**Lowercase value counts**

,club,count
0,robotics,8
1,music,7
2,football,7
3,chess,6
4,chess club,3
5,debating,3
6,soccer,2
7,debate club,1
8,music,1
9,debate,1



2. **Faculty, club, and city:** Map every variant to its canonical value. First, normalise mechanically by stripping whitespace and converting values to lowercase. Then, use an explicit dictionary to handle the remaining variants. For example:
   - `Media Engineering and Technology` → `MET`
   - `Soccer` → `Football`

   Assert that only canonical values remain in these columns.


In [161]:
'''
| `faculty`  | `MET`, `IET`, `EMS`, `Pharmacy`, `Management`      |
| `club`     | `Robotics`, `Debate`, `Music`, `Football`, `Chess` |
| `city`     | `Cairo`, `Giza`, `Alexandria`                      |
'''

FACULTIES = {'met', 'iet', 'ems', 'pharmacy','management'}
CLUBS = {'robotics', 'debate', 'music', 'football', 'chess'}
CITIES = {'cairo', 'giza', 'alexandria'}

faculty = raw["faculty"].str.strip().str.lower()
raw["faculty"] = faculty.replace({"information engineering and technology": "iet",
                              "engineering and materials science":"ems",
                              "pharma":"pharmacy",
                              "media engineering and technology":"met"
                              })

club = raw["club"].str.strip().str.lower()
raw["club"] = club.replace({
                            'chess club':'chess',
                            'debating':'debate',
                            'soccer':'football',
                            'debate club':'debate'
                              })

city = raw["city"].str.strip().str.lower()
raw["city"] = city.replace({
                            'alex':'alexandria',
                            'el giza':'giza',
                              })

In [162]:
assert set(raw["faculty"]) == FACULTIES
raw["faculty"].value_counts()

faculty
iet           12
pharmacy       9
ems            9
met            5
management     4
Name: count, dtype: int64

In [163]:
assert set(raw["club"]) == CLUBS
raw["club"].value_counts()

club
football    9
chess       9
music       8
robotics    8
debate      5
Name: count, dtype: int64

In [164]:
assert set(raw["city"]) == CITIES
raw["city"].value_counts()

city
alexandria    16
cairo         16
giza           7
Name: count, dtype: int64

3. **Names and emails:** Standardise names by removing extra spaces and applying consistent capitalisation. Trim whitespace from email addresses and convert them to lowercase.

In [165]:
raw["email"] = raw["email"].str.strip().str.lower()

In [166]:
raw["full_name"] = raw["full_name"].str.strip().str.title()


4. **`fee_paid`:** Convert inconsistent representations of boolean values into a proper boolean column:
   - `yes`, `Y`, `TRUE`, `1` → `True`
   - `no`, `N`, `false`, `0` → `False`



In [167]:
raw["fee_paid"].value_counts()
fee_paid = raw["fee_paid"].str.strip().str.lower()
fee_paid.value_counts()

fee_paid
yes      11
1         7
false     6
y         5
n         4
true      4
0         1
no        1
Name: count, dtype: int64

In [168]:
mapping = {
    "1": True, "yes": True, "y": True, "true": True,
    "0": False, "no": False, "n": False, "false": False
}

fee_paid = raw["fee_paid"].astype("string").str.strip().str.lower()
raw["fee_paid"] = fee_paid.map(mapping).astype("boolean")

In [169]:
assert set(raw["fee_paid"]) == {True,False}
raw["fee_paid"].value_counts()

fee_paid
True     27
False    12
Name: count, dtype: Int64

5. **`signed_up_at`:** Identify the two date formats mixed in this column. Determine which component represents the day and which represents the month, then parse all values into a single datetime column.

In [170]:
as_text = raw["signed_up_at"].astype(str)
kind = pd.Series(
    np.select(
        [raw["signed_up_at"].map(lambda v: isinstance(v, int)), as_text.str.contains("/", regex=False), as_text.str.endswith("Z")],
        ["epoch ms", "dd/mm local", "ISO, local"],
        default="ISO, local",
    ),
    index=raw.index,
)
raw.groupby(kind)["signed_up_at"].agg(rows="size", example="first")

,rows,example
"ISO, local",25,2026-09-15 13:39
dd/mm local,14,15/09/2026 03:12


In [171]:
dmy = raw.loc[kind == "dd/mm local", "signed_up_at"]
print("first part > 12: ", (dmy.str[:2].astype(int) > 12).sum())
print("second part > 12:", (dmy.str[3:5].astype(int) > 12).sum())

first part > 12:  14
second part > 12: 0


date is either dd/mm/yyyy HH:MM  or yyyy-mm-dd HH:MM

In [172]:
s = raw["signed_up_at"].astype("string").str.strip()

raw["signed_up_at"] = pd.to_datetime(
    s,
    format="mixed",
    dayfirst=True,
    errors="raise"
).dt.strftime("%Y-%m-%d %H:%M")

# Part B — Duplicates

1. **Exact duplicates:** Count rows representing identical form submissions, then remove the duplicate copies.

2. **Repeated sign-ups:** Some students submitted sign-ups for the same club more than once, using different `signup_id` values. These resubmissions typically indicate that a student returned to update their fee payment status. Keep only the latest submission for each combination of `student_id` and `club`. Count the number of rows removed.

3. **Validation:** Assert that the combination of `student_id` and `club` is unique after completing Part B.

Make sure that we do not lose the paid status ?

In [173]:
keys = ["student_id", "club"]
dupes = raw[raw.duplicated(subset=keys, keep=False)]
check = dupes.groupby(keys)["fee_paid"].agg(
    occurrences="size",
    any_paid="any",
    all_paid="all"
)

last_unpaid = dupes.groupby(keys, sort=False).tail(1)
last_unpaid = last_unpaid.set_index(keys)

bad_keys = check[
    check["any_paid"]
].join(last_unpaid["fee_paid"].rename("last_paid"))

bad_keys = bad_keys[
    ~bad_keys["last_paid"].eq(True)
]

print("Duplicated natural keys:", len(check))
print("loses a paid status:", len(bad_keys))
display(bad_keys)

Duplicated natural keys: 7
loses a paid status: 2


,,occurrences,any_paid,all_paid,last_paid
student_id,club,,,,
61-4844,debate,2,True,False,False
61-7344,debate,2,True,False,False


the task does not mention wether we should override the paid status to any paid or not , may be the last one is the correct one ?

In [174]:
dups =raw.duplicated().sum()
raw = raw.drop_duplicates().copy()
raw = raw.drop_duplicates(subset=["student_id", "club"], keep="last").copy()
print('Exact duplicates:',dups)

Exact duplicates: 3


In [175]:
assert raw[["student_id","club"]].duplicated().sum() == 0

In [176]:
raw.describe()

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
count,32,32,32,32,32,32,32,32,32
unique,32,15,17,14,5,3,5,32,2
top,S015,61-7344,Ali Ibrahim,mohamed.adel@student.guc.edu.eg,iet,alexandria,chess,2026-09-15 13:39,True
freq,1,4,4,5,9,14,8,1,24


In [177]:
raw.to_csv('club_signups_cleaned.csv',index=False)